In [ ]:
# train_unsw_nb15_q88.py
# End-to-end: Download UNSW-NB15 → Preprocess → Scale → Train → Quantize (Q8.8)
# → Fixed-point threshold sweep → Export W/b CSVs + manifest.json with (μ,σ, THRESH_Q)

import os, json, csv, glob, sys
from pathlib import Path
import numpy as np
import pandas as pd

# ========= 0) DOWNLOAD DATASET (Kaggle) =========
try:
    import kagglehub  # pip install kagglehub
except ImportError:
    print("Please: pip install kagglehub")
    sys.exit(1)

print("Downloading UNSW-NB15 from Kaggle…")
ds_path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
print("Dataset path:", ds_path)

def find_csv(name_contains):
    for p in glob.glob(os.path.join(ds_path, "**", "*.csv"), recursive=True):
        if name_contains.lower() in os.path.basename(p).lower():
            return p
    return None

train_csv = find_csv("training-set") or find_csv("training")
test_csv  = find_csv("testing-set") or find_csv("testing")

if not train_csv or not test_csv:
    print("Could not find training/testing CSVs under:", ds_path)
    print("Files found:", glob.glob(os.path.join(ds_path, "**", "*.csv"), recursive=True))
    sys.exit(1)

print("Train CSV:", train_csv)
print("Test  CSV:", test_csv)

# ========= 1) LOAD =========
train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

print("\n=== Training columns ===")
print(list(train_df.columns))
print("\n=== Testing columns ===")
print(list(test_df.columns))

# Drop 'id' if present
for df in (train_df, test_df):
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)

# Label column
label_col = 'label' if 'label' in train_df.columns else ('Label' if 'Label' in train_df.columns else None)
if label_col is None:
    print("Could not find label column ('label' or 'Label'). Columns:", list(train_df.columns))
    sys.exit(1)




In [ ]:
# ========= 2) DETERMINISTIC PREPROCESSING (BEFORE training) =========
FEATURES = ["proto","sttl","dttl","swin","dwin","stcpb","dtcpb"]  # 7 features we can parse in P4

# constants
STCPB_DTCPPB_SHIFT = 20           # shrink seq/ack by 2^20
STCPB_DTCPPB_CLAMP = (0, 4095)    # after shrinking, clamp
SWIN_CLAMP = (0, 65535)
DWIN_CLAMP = (0, 65535)
TTL_CLAMP  = (0, 255)

def map_proto_to_ipnum(v):
    if pd.isna(v): return 0
    s = str(v).strip().lower()
    if s == "tcp":  return 6
    if s == "udp":  return 17
    if s == "icmp": return 1
    try:
        return int(s)
    except:
        return 0

def clamp_int(x, lo, hi):
    try:
        xi = int(x)
    except:
        return lo
    return max(lo, min(hi, xi))

def shrink_seq(x):
    try:
        xi = int(x)
    except:
        return 0
    x = x >> STCPB_DTCPPB_SHIFT
    return clamp_int(xi, *STCPB_DTCPPB_CLAMP)

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    if not all(c in df.columns for c in ["proto","sttl","dttl","swin","dwin","stcpb","dtcpb"]):
        missing = [c for c in ["proto","sttl","dttl","swin","dwin","stcpb","dtcpb"] if c not in df.columns]
        raise SystemExit(f"Missing expected columns: {missing}")
    out = pd.DataFrame()
    out["proto"] = df["proto"].apply(map_proto_to_ipnum).astype(int)
    out["sttl"]  = df["sttl"].apply(lambda v: clamp_int(v, *TTL_CLAMP)).astype(int)
    out["dttl"]  = df["dttl"].apply(lambda v: clamp_int(v, *TTL_CLAMP)).astype(int)
    out["swin"]  = df["swin"].apply(lambda v: clamp_int(v, *SWIN_CLAMP)).astype(int)
    out["dwin"]  = df["dwin"].apply(lambda v: clamp_int(v, *DWIN_CLAMP)).astype(int)
    out["stcpb"] = df["stcpb"].apply(shrink_seq).astype(int)
    out["dtcpb"] = df["dtcpb"].apply(shrink_seq).astype(int)
    return out

Xtr_raw = preprocess_df(train_df)
Xte_raw = preprocess_df(test_df)
ytr = train_df[label_col].astype(int).values
yte = test_df[label_col].astype(int).values

# quick preview
print("\n=== Preprocessed preview (train, first 10) ===")
print(Xtr_raw.head(10).to_string(index=False))


In [ ]:
# ========= 3) STANDARDIZE (fit on TRAIN), keep μ,σ =========
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=True, with_std=True)
Xtr = scaler.fit_transform(Xtr_raw[FEATURES])
Xte = scaler.transform(Xte_raw[FEATURES])
scaler_params = {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()}
feature_order = FEATURES[:]  # freeze order

# ========= 4) TRAIN tiny NN (7 → 5 → 5 → 1) =========
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

clf = MLPClassifier(hidden_layer_sizes=(5,5), activation="relu",
                    solver="adam", max_iter=120, random_state=42)
clf.fit(Xtr, ytr)
yp = clf.predict(Xte)
print("\nFloat-domain Accuracy:", accuracy_score(yte, yp))
print(classification_report(yte, yp, digits=4))

# Extract float weights/bias
W1_f, W2_f, W3_f = clf.coefs_[0], clf.coefs_[1], clf.coefs_[2]  # (7x5), (5x5), (5x1)
b1_f, b2_f, b3_f = clf.intercepts_[0], clf.intercepts_[1], clf.intercepts_[2]  # (5,), (5,), (1,)



In [ ]:
# ========= 5) QUANTIZE to Q8.8 (×256) =========
SCALE = 256

def to_q88(x): return int(np.round(float(x) * SCALE))

W1_q = np.vectorize(to_q88)(W1_f).astype(np.int32)              # (7x5)
W2_q = np.vectorize(to_q88)(W2_f).astype(np.int32)              # (5x5)
W3_q = np.vectorize(to_q88)(W3_f).astype(np.int32).reshape(-1)  # (5,)
b1_q = np.vectorize(to_q88)(b1_f).astype(np.int32).reshape(-1)  # (5,)
b2_q = np.vectorize(to_q88)(b2_f).astype(np.int32).reshape(-1)  # (5,)
b3_q = to_q88(b3_f[0])                                          # scalar

# ========= 6) FIXED-POINT INFERENCE + THRESHOLD SWEEP =========
def q88_mul(a_q, x_q):  # (Q8.8 * Q8.8) >> 8
    return np.int32((np.int64(a_q) * np.int64(x_q)) >> 8)

def relu_q(x_q): return np.int32(x_q if x_q > 0 else 0)

def forward_q88_batch(Xfloat, W1_q, b1_q, W2_q, b2_q, W3_q, b3_q):
    """Xfloat: standardized floats; returns logits in Q8.8 int32"""
    X_q = np.round(Xfloat * SCALE).astype(np.int32)  # to Q8.8
    N = X_q.shape[0]

    # Layer 1
    H1 = np.zeros((N,5), dtype=np.int32)
    for j in range(5):
        acc = np.zeros(N, dtype=np.int32)
        for i in range(7):
            acc += q88_mul(W1_q[i, j], X_q[:, i])
        acc += b1_q[j]
        H1[:, j] = np.vectorize(relu_q)(acc)

    # Layer 2
    H2 = np.zeros((N,5), dtype=np.int32)
    for j in range(5):
        acc = np.zeros(N, dtype=np.int32)
        for i in range(5):
            acc += q88_mul(W2_q[i, j], H1[:, i])
        acc += b2_q[j]
        H2[:, j] = np.vectorize(relu_q)(acc)

    # Output
    out = np.zeros(N, dtype=np.int32)
    for i in range(5):
        out += q88_mul(W3_q[i], H2[:, i])
    out += b3_q
    return out  # Q8.8

from sklearn.metrics import f1_score

logits_q = forward_q88_batch(Xte, W1_q, b1_q, W2_q, b2_q, W3_q, b3_q)
cands = np.arange(-2048, 2049, 8, dtype=np.int32)  # ~[-8.0, +8.0] step 0.03125
best_f1, best_thr = -1.0, 0
for t in cands:
    pred = (logits_q >= t).astype(int)
    f1 = f1_score(yte, pred)
    if f1 > best_f1:
        best_f1, best_thr = f1, int(t)
print(f"\n[threshold] best F1={best_f1:.4f} at THRESH_Q={best_thr} (Q8.8 -> {best_thr/256.0:.4f})")

# ========= 7) EXPORT W/b CSVs + manifest.json =========
EXPORT_DIR = Path("nn_export_q88_min7")
EXPORT_DIR.mkdir(exist_ok=True, parents=True)

def write_csv(path, arr, header=None):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        if header is not None:
            w.writerow(header)
        if isinstance(arr, np.ndarray) and arr.ndim == 1:
            for v in arr:
                w.writerow([int(v)])
        else:
            for row in arr:
                w.writerow([int(x) for x in row])

write_csv(EXPORT_DIR/"W1_q.csv", W1_q, header=FEATURES)  # rows=features, cols=5
write_csv(EXPORT_DIR/"b1_q.csv", b1_q)
write_csv(EXPORT_DIR/"W2_q.csv", W2_q)
write_csv(EXPORT_DIR/"b2_q.csv", b2_q)
write_csv(EXPORT_DIR/"W3_q.csv", W3_q)
write_csv(EXPORT_DIR/"b3_q.csv", np.array([b3_q], dtype=np.int32))

with open(EXPORT_DIR/"manifest.json", "w") as f:
    json.dump({
        "feature_order": FEATURES,
        "hidden_sizes": [5,5],
        "scale": "Q8.8 (x256)",
        "scaler": scaler_params,
        "threshold_q88": best_thr,
        "note": "Preprocess BEFORE training; runtime tables must apply the same StandardScaler (μ,σ) offline to bucket reps."
    }, f, indent=2)

print(f"\nExported quantized model & manifest to: {EXPORT_DIR.resolve()}")
print("Next: run your gen_runtime_from_q88.py to build s1-runtime.json using these files.")


## gen_runtime_from_q88.py
Goal: Turn your quantized model + bucket plans + scaler params into a BMv2 runtime JSON (s1-runtime.json) that precomputes all integer products and loads biases/threshold—so the switch only does adds and compares.

In [ ]:
# ['proto','sttl','dttl','swin','dwin','stcpb','dtcpb']

import numpy as np, json, os
os.makedirs("/content/prepped", exist_ok=True)

def qtiles(col, qs):
    vals = np.unique(np.quantile(col, qs).astype(int))
    return vals.tolist()

bucket_plan = {
    "proto":  sorted([1,6,17] + [int(v) for v in np.unique(Xtr_raw['proto']) if v in (1,6,17)])[:3],
    "sttl":   qtiles(Xtr_raw['sttl'],  [0.2, 0.5, 0.8, 0.98]),
    "dttl":   qtiles(Xtr_raw['dttl'],  [0.0, 0.5, 0.9]),
    "swin":   qtiles(Xtr_raw['swin'],  [0.0, 0.6, 0.95]),
    "dwin":   qtiles(Xtr_raw['dwin'],  [0.0, 0.9]),
    "stcpb":  qtiles(Xtr_raw['stcpb'], [0.0, 0.5, 0.7, 0.85, 0.95]),
    "dtcpb":  qtiles(Xtr_raw['dtcpb'], [0.0, 0.5, 0.7, 0.85, 0.95]),
}
with open("/content/prepped/bucket_plan.json","w") as f:
    json.dump(bucket_plan, f, indent=2)

print("Wrote /content/prepped/bucket_plan.json from data")


In [ ]:
# gen_runtime_from_q88.py
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd

# ===================== CONFIG (change paths if needed) =====================
EXPORT_DIR = Path("/content/nn_export_q88_min7")   # W1_q.csv, b1_q.csv, W2_q.csv, b2_q.csv, W3_q.csv, b3_q.csv, manifest.json
BUCKET_PLAN = Path("/content/prepped/bucket_plan.json")
OUT_JSON    = Path("/content/ids-topo/s1-runtime.json")

# Default feature order if manifest is missing:
DEFAULT_FEATURE_ORDER = ["proto","sttl","dttl","swin","dwin","stcpb","dtcpb"]

# H1/H2 bucket representatives (post ReLU and >>FRAC_BITS in P4)
H1_BUCKETS = [0, 1, 2, 3, 5, 10, 20, 40]
H2_BUCKETS = [0, 1, 2, 3, 5, 10]

# Threshold in Q8.8 (tune later)
THRESH_Q = 200

# IPv4 LPM entries to keep (edit to match your topo)
LPM = [
    {"hdr.ipv4.dstAddr": ["10.0.1.1", 32], "dstAddr": "08:00:00:00:01:11", "port": 1},
    {"hdr.ipv4.dstAddr": ["10.0.2.2", 32], "dstAddr": "08:00:00:00:02:22", "port": 2},
]
# ==========================================================================


def die(msg):
    print("[error]", msg)
    sys.exit(1)

def mul_q88(a_q, x_q):
    """(Q8.8 * Q8.8) >> 8 -> Q8.8, saturated to int32."""
    prod = (int(a_q) * int(x_q)) >> 8
    if prod >  2**31 - 1: prod =  2**31 - 1
    if prod < -2**31:     prod = -2**31
    return int(prod)

def q88_from_float(x):
    return int(round(float(x) * 256.0))

def q88_from_bucket(value, mean, scale):
    """Apply StandardScaler then convert to Q8.8."""
    if scale == 0:  # guard
        xprime = 0.0
    else:
        xprime = (float(value) - float(mean)) / float(scale)
    return q88_from_float(xprime)

def load_csv_matrix_simple(path):
    df = pd.read_csv(path, header=None)
    return df.values.astype(np.int64)

def load_W1_robust(path, expected_cols=5):
    """
    Robust reader for W1_q.csv:
    - Accepts files with/without header
    - If the first column is non-numeric (feature names), drops it
    - Ensures final shape is (n_features, expected_cols)
    """
    df0 = pd.read_csv(path)  # try with header if present
    # First, try coercion
    df_num = df0.apply(pd.to_numeric, errors='coerce')
    # Drop fully-empty rows/cols
    df_num = df_num.dropna(axis=1, how='all').dropna(axis=0, how='all')
    # If after coercion we still don't have expected columns, try dropping the first original col (likely index)
    if df_num.shape[1] != expected_cols and df0.shape[1] >= expected_cols + 1:
        try:
            df_num2 = pd.to_numeric(df0.iloc[:, 1:], errors='coerce')
            df_num2 = df_num2.dropna(axis=1, how='all').dropna(axis=0, how='all')
            if df_num2.shape[1] == expected_cols:
                df_num = df_num2
        except Exception:
            pass
    if df_num.shape[1] != expected_cols:
        raise ValueError(f"W1_q.csv parsing error: got {df_num.shape[1]} columns, expected {expected_cols}. "
                         f"Tip: export with columns H1_0..H1_4 and feature names as index, or no header at all.")
    return df_num.values.astype(np.int64)

# -------- Load weights/biases --------
W1_path = EXPORT_DIR / "W1_q.csv"
b1_path = EXPORT_DIR / "b1_q.csv"
W2_path = EXPORT_DIR / "W2_q.csv"
b2_path = EXPORT_DIR / "b2_q.csv"
W3_path = EXPORT_DIR / "W3_q.csv"
b3_path = EXPORT_DIR / "b3_q.csv"
man_path= EXPORT_DIR / "manifest.json"

for p in [W1_path,b1_path,W2_path,b2_path,W3_path,b3_path]:
    if not p.exists(): die(f"Missing file: {p}")

try:
    W1 = load_W1_robust(W1_path)                 # (nfeat, 5)
except Exception as e:
    die(f"Failed to parse {W1_path}: {e}")

b1 = load_csv_matrix_simple(b1_path).reshape(-1) # (5,)
W2 = load_csv_matrix_simple(W2_path)             # (5,5)
b2 = load_csv_matrix_simple(b2_path).reshape(-1) # (5,)
W3 = load_csv_matrix_simple(W3_path).reshape(-1) # (5,)
b3 = int(load_csv_matrix_simple(b3_path).reshape(-1)[0])

# -------- Manifest / scaler / feature order --------
if man_path.exists():
    mj = json.loads(man_path.read_text())
    feat_order = mj.get("feature_order", DEFAULT_FEATURE_ORDER)
    scaler = mj.get("scaler", {})
    means  = scaler.get("mean", [0.0]*len(feat_order))
    scales = scaler.get("scale", [1.0]*len(feat_order))
    print(f"[info] loaded manifest.json; feature_order={feat_order}")
else:
    feat_order = DEFAULT_FEATURE_ORDER
    means  = [0.0]*len(feat_order)
    scales = [1.0]*len(feat_order)
    print("[warn] manifest.json not found; assuming NO SCALING and default feature order.")

# Sanity: shapes
nfeat = len(feat_order)
if W1.shape != (nfeat, 5):
    die(f"W1 shape {W1.shape} does not match expected ({nfeat}, 5). Check feature_order vs rows in W1_q.csv.")
if W2.shape != (5,5):
    die(f"W2 shape {W2.shape} expected (5,5)")
if len(b1) != 5 or len(b2) != 5 or len(W3) != 5:
    die("b1/b2/W3 shapes off; expected lengths 5.")

# -------- Bucket plan --------
if not BUCKET_PLAN.exists():
    die(f"Missing bucket plan: {BUCKET_PLAN}")
bucket_plan = json.loads(BUCKET_PLAN.read_text())

# -------- Build table entries --------
entries = []

# IPv4 LPM (default drop + forwards)
entries.append({
    "table": "MyIngress.ipv4_lpm",
    "default_action": True,
    "action_name": "MyIngress.drop",
    "action_params": {}
})
for row in LPM:
    entries.append({
        "table": "MyIngress.ipv4_lpm",
        "match": { "hdr.ipv4.dstAddr": row["hdr.ipv4.dstAddr"] },
        "action_name": "MyIngress.ipv4_forward",
        "action_params": { "dstAddr": row["dstAddr"], "port": row["port"] }
    })

# Feature -> H1 cloned tables: t_f_<feat>_to_h1_<h>
for fi, feat in enumerate(feat_order):
    reps = bucket_plan.get(feat, None)
    if reps is None:
        print(f"[warn] feature '{feat}' missing in bucket_plan.json; skipping its tables.")
        continue
    mean  = float(means[fi]) if fi < len(means)  else 0.0
    scale = float(scales[fi]) if fi < len(scales) else 1.0

    for h in range(5):
        tname = f"MyIngress.t_f_{feat}_to_h1_{h}"
        w_q = int(W1[fi, h])
        for val in reps:
            x_q = q88_from_bucket(val, mean, scale)   # apply same StandardScaler as training
            prod_q32 = mul_q88(w_q, x_q)
            entries.append({
                "table": tname,
                "match": { f"meta.{feat}": int(val) },
                "action_name": f"MyIngress.add_to_h1_{h}",
                "action_params": { "prod_q32": int(prod_q32) }
            })

# H1 -> H2 cloned tables: t_h1_<i>_to_h2_<j>, key=meta.h1_bucket<i>
for i in range(5):
    for j in range(5):
        tname = f"MyIngress.t_h1_{i}_to_h2_{j}"
        w_q = int(W2[i, j])
        for hb in H1_BUCKETS:
            hb_q = q88_from_float(hb)  # H1 buckets are already post-activation magnitudes; no scaler
            prod_q32 = mul_q88(w_q, hb_q)
            entries.append({
                "table": tname,
                "match": { f"meta.h1_bucket{i}": int(hb) },
                "action_name": f"MyIngress.add_to_h2_{j}",
                "action_params": { "prod_q32": int(prod_q32) }
            })

# H2 -> OUT tables: t_h2_<j>_to_out, key=meta.h2_bucket<j>
for j in range(5):
    tname = f"MyIngress.t_h2_{j}_to_out"
    w_q = int(W3[j])
    for hb in H2_BUCKETS:
        hb_q = q88_from_float(hb)  # H2 buckets too
        prod_q32 = mul_q88(w_q, hb_q)
        entries.append({
            "table": tname,
            "match": { f"meta.h2_bucket{j}": int(hb) },
            "action_name": "MyIngress.add_to_out",
            "action_params": { "prod_q32": int(prod_q32) }
        })

# Registers for biases and threshold
registers = []
for idx, v in enumerate(b1): registers.append({"name":"MyIngress.reg_b1_q", "index":idx, "value":int(v)})
for idx, v in enumerate(b2): registers.append({"name":"MyIngress.reg_b2_q", "index":idx, "value":int(v)})
registers.append({"name":"MyIngress.reg_b3_q",    "index":0, "value":int(b3)})
registers.append({"name":"MyIngress.reg_thresh_q","index":0, "value":int(THRESH_Q)})

# Final JSON object
obj = {
    "target": "bmv2",
    "p4info": "build/nn_ids.p4.p4info.txtpb",
    "bmv2_json": "build/nn_ids.json",
    "table_entries": entries,
    "register_entries": registers
}

# Write and offer download
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUT_JSON.write_text(json.dumps(obj, indent=2))
print(f"[ok] wrote {OUT_JSON} with {len(entries)} table entries and {len(registers)} register writes.")

# In Colab: trigger download
try:
    from google.colab import files
    files.download(str(OUT_JSON))
except Exception:
    pass
